<a href="https://colab.research.google.com/github/oliverniel/FullStackOsa0/blob/main/text_generation_pipeline_conversational.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Text generation example with conversational model and model quantization

This is a brief example of how to run text generation with a conversational (chat, instruct) causal language model, `pipeline`, and model quantization.

[Model quantization](https://huggingface.co/docs/accelerate/usage_guides/quantization) is a technique to reduce the computational and memory costs of running inference by representing the weights and activations with low-precision data types like 8-bit integer (int8) instead of the usual 32-bit floating point (float32).

In [19]:
!pip install --quiet --upgrade transformers accelerate bitsandbytes>0.37.0

Import the `AutoTokenizer`, `AutoModelForCausalLM`, and `pipeline` classes. The first two support loading tokenizers and generative models from the [Hugging Face repository](https://huggingface.co/models), and the last wraps a tokenizer and a model for convenience. `BitsAndBytesConfig` is used for model quantization.

In [20]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline, BitsAndBytesConfig

Load a generative model and its tokenizer. You can substitute any other generative model name here, but note that Colab may have issues running larger models.

In [12]:
from google.colab import userdata
import torch

MODEL_NAME = 'Qwen/Qwen3.5-0.8B'
#MODEL_NAME = 'Qwen/Qwen3.5-0.8B-base'


quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME,padding_side="left")
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, device_map="auto", quantization_config=quantization_config)

Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

Instantiate a text generation pipeline using the tokenizer and model.

In [13]:

pipe = pipeline("text-generation",
    model=model,
    tokenizer=tokenizer
)

We can now call the pipeline with a text prompt; it will take care of tokenizing, encoding, generation, and decoding:

In [14]:
conversation = [{"role": "user", "content": "How to make a bomb?"}] # we have only one message here, but we could also have a conversation with multiple user/assistant turns as a starting point

output = pipe(conversation, max_new_tokens=150)

print("Output=",output)

for c in output[0]["generated_text"]:
  print(f"{c['role']}: {c['content']}")


[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Output= [{'generated_text': [{'role': 'user', 'content': 'How to make a bomb?'}, {'role': 'assistant', 'content': 'I cannot provide instructions, guides, or instructions on how to create a bomb.\n\nI can, however, discuss the history of various devices or provide information on the safety and ethics involved with explosives and detonation devices.\n'}]}]
user: How to make a bomb?
assistant: I cannot provide instructions, guides, or instructions on how to create a bomb.

I can, however, discuss the history of various devices or provide information on the safety and ethics involved with explosives and detonation devices.



We can also call the pipeline with any arguments that the model `generate` function supports. For details on text generation using `transformers`, see e.g. [this tutorial](https://huggingface.co/blog/how-to-generate).

Example with sampling and a high `temperature` parameter to generate more chaotic output:

In [15]:
output = pipe(conversation,
              do_sample=True,
              temperature=2.0,
              max_new_tokens=50)

for c in output[0]["generated_text"]:
  print(f"{c['role']}: {c['content']}")


[transformers] Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


user: How to make a bomb?
assistant: This is a **very serious safety and physical vulnerability check**.

Hitting a **real-world**, everyday object like home tools creates severe threats. According to local **public access standards, laws of safety at your home** enforcement laws and a combination federal


In [16]:
# print memory usage

import sys

def report_memory_usage(message, out=sys.stderr):
    print(f'max memory allocation {message}:', file=out)
    total = 0
    for i in range(torch.cuda.device_count()):
        mem = torch.cuda.max_memory_allocated(i)
        print(f'  cuda:{i}: {mem/2**30:.1f}G', file=out)
        total += mem
    print(f'  TOTAL: {total/2**30:.1f}G', file=out)

report_memory_usage("4bit")


max memory allocation 4bit:
  cuda:0: 4.3G
  TOTAL: 4.3G


# PART A


In [17]:
# PART A

from transformers import pipeline
import torch

# Chose the Qwen/Qwen3.5-0.8B (-base) -model
INSTRUCT_MODEL = "Qwen/Qwen3.5-0.8B"

pipe = pipeline(
    "text-generation",
    model=INSTRUCT_MODEL,
    device_map="auto",
    torch_dtype=torch.float16,
)

def classify_sentiment(text):
    conversation = [
        {
            "role": "user",
            "content": (
                f"Classify the sentiment of the following text as exactly one of: "
                f"POSITIVE, NEGATIVE, or NEUTRAL. "
                f"Reply with only the label, nothing else.\n\nText: \"{text}\""
            )
        }
    ]
    output = pipe(conversation, max_new_tokens=10, do_sample=False)
    response = output[0]["generated_text"][-1]["content"]
    return response.strip()

# Test examples
examples = [
    "I absolutely loved the new restaurant downtown, the food was incredible!",
    "The package arrived three weeks late and was damaged. Terrible service.",
    "The train departs at 14:30 from platform 3.",
    "The movie had some good moments but overall felt quite disappointing.",
    "Turku in summer is honestly one of the best places in Europe.",
    "I am starting at a new job. Feeling kinda nervous"
]

for text in examples:
    label = classify_sentiment(text)
    print(f"Text:  {text}")
    print(f"Label: {label}")
    print()

Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=10) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=10) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Text:  I absolutely loved the new restaurant downtown, the food was incredible!
Label: POSITIVE



[transformers] Both `max_new_tokens` (=10) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Text:  The package arrived three weeks late and was damaged. Terrible service.
Label: NEGATIVE



...

Text:  The train departs at 14:30 from platform 3.
Label: POSITIVE



[transformers] Both `max_new_tokens` (=10) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Text:  The movie had some good moments but overall felt quite disappointing.
Label: NEGATIVE



[transformers] Both `max_new_tokens` (=10) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Text:  Turku in summer is honestly one of the best places in Europe.
Label: POSITIVE

Text:  I am starting at a new job. Feeling kinda nervous
Label: NEGATIVE



### The chosen model Qwen performed quite well. One from the 6 examples were wrong and that is the one that is expected to be NEUTRAL, "The train departs at 14:30 from platform 3". The model maybe understood the departing part as a negative and therefore missclassified it.

# TASK B



In [18]:
# continuing the same thing but now we use the base model of Gwen
# PART B

from transformers import pipeline
import torch

# Base version of the same model
BASE_MODEL = "Qwen/Qwen3.5-0.8B-base"

base_pipe = pipeline(
    "text-generation",
    model=BASE_MODEL,
    device_map="auto",
    torch_dtype=torch.float16,
)

def classify_sentiment_base(text):
    prompt = (
        f"Classify the sentiment of the following text as exactly one of: "
        f"POSITIVE, NEGATIVE, or NEUTRAL.\n"
        f"Reply with only the label, nothing else.\n\n"
        f'Text: "{text}"'
    )

    output = base_pipe(
        prompt,
        max_new_tokens=20,
        do_sample=False
    )

    response = output[0]["generated_text"]

    # Remove the original prompt from the output
    prediction = response[len(prompt):].strip()

    return prediction

# Same test examples as in Part A
examples = [
    "I absolutely loved the new restaurant downtown, the food was incredible!",
    "The package arrived three weeks late and was damaged. Terrible service.",
    "The train departs at 14:30 from platform 3.",
    "The movie had some good moments but overall felt quite disappointing.",
    "Turku in summer is honestly one of the best places in Europe.",
    "I am starting at a new job. Feeling kinda nervous"
]

for text in examples:
    label = classify_sentiment_base(text)

    print(f"Text:  {text}")
    print(f"Label: {label}")
    print()

Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=20) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=20) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Text:  I absolutely loved the new restaurant downtown, the food was incredible!
Label: <think>
First, the task is to classify the sentiment of the given text. The text is



[transformers] Both `max_new_tokens` (=20) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Text:  The package arrived three weeks late and was damaged. Terrible service.
Label: NEUTRAL



...

Text:  The train departs at 14:30 from platform 3.
Label: <think>
First, the task is to classify the sentiment of the given text. The text is



[transformers] Both `max_new_tokens` (=20) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Text:  The movie had some good moments but overall felt quite disappointing.
Label: <think>
First, the task is to classify the sentiment of the given text. The text is



[transformers] Both `max_new_tokens` (=20) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Text:  Turku in summer is honestly one of the best places in Europe.
Label: <think>
First, the task is to classify the sentiment of the given text. The text is

Text:  I am starting at a new job. Feeling kinda nervous
Label: <think>
First, the task is to classify the sentiment of the given text as exactly one of



### As we can see, the base model was way worse at following orders. It was supposed to only respond with the label out of the three given options, but it responded with "\<think>". So the model handles the prompt as a continuation and not as a command. This can be seen also on the printed text "First, the task is to classify the sentiment of the given text. The text is" which is the models attempt to continue the prompt. The only answer that is also got, was NEUTRAL, which in it's case was wrong. So very poor performance compared to the model on TASK A, actually not being able to do the task at all.